# Gold — Season Progression
Progressão corrida a corrida dos pilotos dentro de cada temporada, com métricas cumulativas calculadas via window functions.

In [ ]:
import dlt
from pyspark.sql.functions import (
    col, sum, when, rank, first
)
from pyspark.sql.window import Window

spark.sql("USE CATALOG f1_lakehouse")

In [ ]:
@dlt.table(
    name="season_progression",
    comment="Progressão dos pilotos corrida a corrida com métricas cumulativas",
    table_properties={"quality": "gold"},
    partition_cols=["season"]
)
@dlt.expect_all({
    "valid_driver_id": "driver_id IS NOT NULL",
    "valid_round":     "round IS NOT NULL",
    "valid_season":    "season IS NOT NULL"
})
def season_progression():
    # Flags por corrida para acumulação
    df = (
        dlt.read("results")
        .select(
            col("season"),
            col("race_round").cast("int").alias("round"),
            col("race_name"),
            col("race_date"),
            col("driver_id"),
            col("driver_name"),
            col("constructor_id"),
            col("points").alias("points_in_race"),
            when(col("final_position") == 1, 1).otherwise(0).alias("_is_win"),
            when(col("final_position") <= 3, 1).otherwise(0).alias("_is_podium"),
            when(
                col("position_text").isin("R", "D", "E", "W", "F", "N"), 1
            ).otherwise(0).alias("_is_dnf")
        )
    )

    # Window cumulativa por piloto dentro da temporada
    driver_cum_window = (
        Window
        .partitionBy("season", "driver_id")
        .orderBy("round")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )

    df = (
        df
        .withColumn("cumulative_points", sum("points_in_race").over(driver_cum_window))
        .withColumn("wins_so_far",        sum("_is_win").over(driver_cum_window))
        .withColumn("podiums_so_far",     sum("_is_podium").over(driver_cum_window))
        .withColumn("dnfs_so_far",        sum("_is_dnf").over(driver_cum_window))
    )

    # Posição no campeonato após cada rodada
    # Desempate: mais vitórias acumuladas
    round_rank_window = (
        Window
        .partitionBy("season", "round")
        .orderBy(col("cumulative_points").desc(), col("wins_so_far").desc())
    )
    df = df.withColumn("championship_position", rank().over(round_rank_window))

    # Gap para o líder naquela rodada
    leader_window = (
        Window
        .partitionBy("season", "round")
        .orderBy(col("cumulative_points").desc())
    )
    df = (
        df
        .withColumn(
            "_leader_points",
            first("cumulative_points", ignorenulls=True).over(leader_window)
        )
        .withColumn("gap_to_leader", col("_leader_points") - col("cumulative_points"))
        .drop("_leader_points", "_is_win", "_is_podium", "_is_dnf")
    )

    return (
        df
        .select(
            "season",
            "round",
            "race_name",
            "race_date",
            "driver_id",
            "driver_name",
            "constructor_id",
            "points_in_race",
            "cumulative_points",
            "championship_position",
            "wins_so_far",
            "podiums_so_far",
            "dnfs_so_far",
            "gap_to_leader"
        )
        .orderBy("season", "round", "championship_position")
    )